In [ ]:
import sys

sys.path.append("..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from AnimalCLEF_baselines.metrics import score_2025, score_2026
from AnimalCLEF_baselines.submissions import evaluate_submissions, get_submission, load_submissions, select_best
from matplotlib.collections import PathCollection
from matplotlib.ticker import FuncFormatter

In [ ]:
def count_individuals_true(solution: pd.DataFrame) -> dict:
    return solution.groupby("dataset")["identity"].nunique().to_dict()


def overshoot_individuals_2026(submission: pd.DataFrame, solution: pd.DataFrame) -> dict:
    n_true = count_individuals_true(solution)
    clusters = pd.Series(submission["cluster"].to_numpy(), index=solution.index)
    n_pred = clusters.groupby(solution["dataset"]).nunique().to_dict()
    return {dataset: n_pred[dataset] / n_true[dataset] for dataset in n_pred}


def overshoot_individuals_2025(submission: pd.DataFrame, solution: pd.DataFrame) -> dict:
    n_true = count_individuals_true(solution)
    predictions = submission["identity"].to_numpy()
    overshoot = {}
    for dataset in solution["dataset"].unique():
        mask = (solution["dataset"] == dataset).to_numpy()
        predictions_dataset = predictions[mask]
        is_known = predictions_dataset != "new_individual"
        n_known = is_known.sum()
        if n_known == 0:
            overshoot[dataset] = np.nan
            continue
        n_returning = len(set(predictions_dataset[is_known]))
        n_pred = n_returning * len(predictions_dataset) / n_known
        overshoot[dataset] = n_pred / n_true[dataset]
    return overshoot


def overshoot_newcomers_2025(submission: pd.DataFrame, solution: pd.DataFrame) -> dict:
    n_true = solution.groupby("dataset")["new_identity"].sum().to_dict()

    predictions = submission["identity"].to_numpy()
    overshoot = {}
    for dataset in solution["dataset"].unique():
        mask = (solution["dataset"] == dataset).to_numpy()
        predictions_dataset = predictions[mask]
        n_pred = (predictions_dataset == "new_individual").sum()
        if n_true[dataset] == 0:
            overshoot[dataset] = np.nan
            continue
        overshoot[dataset] = n_pred / n_true[dataset]
    return overshoot

In [ ]:
editions = {
    "AnimalCLEF2025": {
        "root": "/data/wildlife_datasets/data/AnimalCLEF2025",
        "score": score_2025,
        "metric": "normalized",
        "replace": {"0009": "XiangYuhang", "0011": "XuZhe"},
        "count_fn": overshoot_individuals_2025,
        "plot_fn": overshoot_newcomers_2025,
        "plot_name": "Open-set",
        "score_col": "{}_normalized",
        "y_label": "ratio of predicted/true newcomers",
    },
    "AnimalCLEF2026": {
        "root": "/data/wildlife_datasets/data/AnimalCLEF2026",
        "score": score_2026,
        "metric": "mean",
        "replace": {},
        "count_fn": overshoot_individuals_2026,
        "plot_fn": None,
        "plot_name": "Clustering",
        "score_col": "{}",
        "y_label": None,
    },
}

cols = ["LynxID2025", "SalamanderID2025", "SeaTurtleID2022"]
plot_names = {
    "LynxID2025": "Lynx",
    "SalamanderID2025": "Fire salamander",
    "SeaTurtleID2022": "Sea turtle",
}

In [ ]:
solutions_all = {}
submissions_all = {}
results_all = {}
for name, edition in editions.items():
    root = edition["root"]

    solution = pd.read_csv(f"{root}/solution.csv")
    idx_public = solution["Usage"] == "Public"
    idx_private = ~idx_public

    root_submission = f"{root}/submissions"
    submissions = load_submissions(root_submission, participants_replace=edition["replace"])
    public, private = evaluate_submissions(submissions, solution, edition["score"], idx_public)
    results = select_best(submissions, public, private, metric=edition["metric"])

    solutions_all[name] = solution
    submissions_all[name] = submissions
    results_all[name] = results

In [ ]:
rows = []
for name, edition in editions.items():
    solution = solutions_all[name]
    submissions = submissions_all[name]
    results = results_all[name]
    estimator = edition["count_fn"]
    plot_fn = edition["plot_fn"]
    ranks = {}
    for col in cols:
        scores = results[edition["score_col"].format(col)].astype(float)
        ranks[col] = scores.rank(ascending=False, method="min")
    for i in range(len(results)):
        submission = get_submission(results, submissions, i)
        overshoot_individuals = estimator(submission, solution)
        overshoot_newcomers = {}
        if plot_fn is not None:
            overshoot_newcomers = plot_fn(submission, solution)
        for dataset in cols:
            rows.append(
                {
                    "edition": name,
                    "team": results.index[i],
                    "rank": ranks[dataset].iloc[i],
                    "dataset": dataset,
                    "individuals": overshoot_individuals[dataset],
                    "newcomers": overshoot_newcomers.get(dataset, np.nan),
                }
            )
estimates = pd.DataFrame(rows)

In [ ]:
def plot_ranked(
    ax: plt.Axes, df: pd.DataFrame, y: str, offset: float, marker: str, label: str, norm: plt.Normalize
) -> PathCollection:
    df = df.sort_values("rank", ascending=False)
    rank_max = df.groupby("dataset")["rank"].transform("max")
    rank_rel = (df["rank"] - 1) / (rank_max - 1)
    x = df["dataset"].map(x_positions) + offset
    x = x + (rank_rel - 0.5) * 0.25
    return ax.scatter(
        x,
        df[y],
        c=df["rank"],
        cmap="viridis_r",
        norm=norm,
        marker=marker,
        edgecolors="black",
        linewidths=0.3,
        label=label,
    )


df = estimates.dropna(subset=["individuals"])
df_new = estimates.dropna(subset=["newcomers"])

x_positions = {col: i for i, col in enumerate(cols)}
offsets = {"AnimalCLEF2025": -0.2, "AnimalCLEF2026": 0.2}
markers = {"AnimalCLEF2025": "o", "AnimalCLEF2026": "s"}
norm = plt.Normalize(estimates["rank"].min(), estimates["rank"].max())
ylims = [(0.25, 16), (0.5, 4)]
yticks = [(0.25, 0.5, 1, 2, 4, 8, 16), (0.5, 1, 2, 4)]

fig, axes = plt.subplots(2, 1, figsize=(7, 5), sharex=True)
for name, edition in editions.items():
    df_edition = df[df["edition"] == name]
    sc = plot_ranked(axes[0], df_edition, "individuals", offsets[name], markers[name], edition["plot_name"], norm)
plot_ranked(
    axes[1],
    df_new,
    "newcomers",
    offsets["AnimalCLEF2025"],
    markers["AnimalCLEF2025"],
    editions["AnimalCLEF2025"]["plot_name"],
    norm,
)
fig.colorbar(sc, ax=axes, label="Rank")

axes[0].set_ylabel("Predicted/true individuals")
axes[1].set_ylabel("Predicted/true newcomers")
for ax, ylim, ytick in zip(axes, ylims, yticks):
    ax.axhline(1, linestyle="--", color="red")
    ax.set_yscale("log", base=2)
    if ytick is not None:
        ax.set_yticks(ytick)
        ax.minorticks_off()
    ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{y:g}"))
    ax.set_ylim(ylim)
    ax.grid(alpha=0.4)
legend = axes[0].legend()
for handle in legend.legend_handles:
    handle.set_facecolor("lightgray")
axes[1].set_xticks(range(len(cols)), [plot_names[col] for col in cols])
plt.savefig("population_editions.png", dpi=300, bbox_inches="tight")